<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/lab-05-model-bake-off-for-orbit-retail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5 (graded) — Model bake-off for Orbit Retail
**Course 2: Generative AI and LLMs with Python — Chapter 5: Foundation models in practice**

**Problem brief (Leo Farkas, Orbit Retail):** "We want auto-generated product descriptions
and weekly review digests. Which model — and what will it cost at our volume?"

**What you'll submit:** at least 3 models evaluated (at least 1 hosted, at least 1 local) on
both tasks, a cost/latency table, and a recommendation with justification.

In [ ]:
!pip install -q transformers openai

## 1. Data: real Amazon-style product reviews (with offline fallback)

In [ ]:
def load_reviews():
    try:
        from datasets import load_dataset
        ds = load_dataset('amazon_polarity', split='train[:200]')
        print('Loaded a real Amazon reviews sample.')
        return [{'title': r['title'], 'content': r['content']} for r in ds]
    except Exception as e:
        print(f'Offline fallback engaged ({e}).')
        return [
            {'title': 'Great sound quality', 'content': 'These headphones have amazing bass and the battery lasts all day. Comfortable for long wear.'},
            {'title': 'Disappointing', 'content': 'The build quality feels cheap and it broke after two weeks. Would not recommend.'},
            {'title': 'Good value', 'content': 'Does what it says, decent quality for the price. Shipping was fast.'},
        ]

reviews = load_reviews()
product_task_prompt = (
    'Write a concise, appealing 2-sentence product description for wireless headphones '
    'based on these customer reviews:\n' + '\n'.join(f'- {r["content"][:150]}' for r in reviews[:5])
)
digest_task_prompt = (
    'Summarize the key themes across these customer reviews in 3 bullet points:\n' +
    '\n'.join(f'- {r["content"][:150]}' for r in reviews[5:15])
)
tasks = {'product_description': product_task_prompt, 'review_digest': digest_task_prompt}

## 2. Define the candidates: local models + optional hosted API

In [ ]:
import time
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

device_idx = 0 if torch.cuda.is_available() else -1


class _Seq2SeqPipe:
    """Pipeline-call-compatible wrapper for encoder-decoder models like flan-t5: newer
    transformers releases dropped the text2text-generation pipeline task and the
    Text2TextGenerationPipeline class entirely, so we call generate() directly instead."""
    task = 'text2text-generation'

    def __init__(self, model_name, device=-1):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.device = torch.device('cuda') if device >= 0 else torch.device('cpu')
        self.model.to(self.device)

    def __call__(self, prompt, max_new_tokens=80, **kwargs):
        ids = self.tokenizer(prompt, return_tensors='pt', truncation=True).input_ids.to(self.device)
        out = self.model.generate(ids, max_new_tokens=max_new_tokens)
        return [{'generated_text': self.tokenizer.decode(out[0], skip_special_tokens=True)}]


local_candidates = {
    'distilgpt2 (local, 82M)': {'pipe': pipeline('text-generation', model='distilgpt2', device=device_idx),
                                  'cost_per_1k_tokens_usd': 0.0, 'kind': 'local'},
    'flan-t5-small (local, 80M)': {'pipe': _Seq2SeqPipe('google/flan-t5-small', device=device_idx),
                                     'cost_per_1k_tokens_usd': 0.0, 'kind': 'local'},
}

import os
try:
    from google.colab import userdata
    API_KEY = userdata.get('LLM_API_KEY')
    BASE_URL = userdata.get('LLM_BASE_URL')
except Exception:
    API_KEY = os.environ.get('LLM_API_KEY')
    BASE_URL = os.environ.get('LLM_BASE_URL')

hosted_available = bool(API_KEY and BASE_URL)
print('Hosted candidate available:', hosted_available)

## 3. Run every candidate on every task, timing + logging output

In [ ]:
results = []

def run_local(name, cfg, task_name, prompt):
    t0 = time.perf_counter()
    if 'text2text' in cfg['pipe'].task:
        out = cfg['pipe'](prompt, max_new_tokens=80)[0]['generated_text']
    else:
        out = cfg['pipe'](prompt, max_new_tokens=80, do_sample=True, top_p=0.9)[0]['generated_text'][len(prompt):]
    latency = time.perf_counter() - t0
    return {'model': name, 'task': task_name, 'output': out.strip(), 'latency_s': round(latency, 2),
            'est_cost_usd': 0.0}

def run_hosted(task_name, prompt, model_name='llama-3.1-8b-instant', price_per_1k=0.0002):
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=model_name, messages=[{'role': 'user', 'content': prompt}], max_tokens=100,
    )
    latency = time.perf_counter() - t0
    tokens = resp.usage.total_tokens
    return {'model': f'{model_name} (hosted)', 'task': task_name,
            'output': resp.choices[0].message.content.strip(),
            'latency_s': round(latency, 2), 'est_cost_usd': round(tokens / 1000 * price_per_1k, 6)}

for task_name, prompt in tasks.items():
    for name, cfg in local_candidates.items():
        results.append(run_local(name, cfg, task_name, prompt))
    if hosted_available:
        try:
            results.append(run_hosted(task_name, prompt))
        except Exception as e:
            print(f'Hosted call failed ({e}) — continuing with local candidates only.')

import pandas as pd
results_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 80)
results_df

## 4. Cost/latency table at Orbit's real volume

In [ ]:
ORBIT_MONTHLY_REQUESTS = 50_000  # both tasks combined, at Orbit's stated volume

cost_table = (
    results_df.groupby('model')
    .agg(avg_latency_s=('latency_s', 'mean'), avg_cost_per_call_usd=('est_cost_usd', 'mean'))
    .reset_index()
)
cost_table['est_monthly_cost_usd'] = (cost_table['avg_cost_per_call_usd'] * ORBIT_MONTHLY_REQUESTS).round(2)
cost_table['est_monthly_compute_hours_if_local'] = (
    cost_table['avg_latency_s'] * ORBIT_MONTHLY_REQUESTS / 3600
).round(1)
cost_table

## 5. Recommendation (fill in)
Using the quality you can see in `results_df` and the cost/latency table above, apply the
model-selection rubric from the chapter (quality / cost / latency / context / license /
privacy) and recommend one model for each of Orbit's two tasks. They don't have to be the
same model.

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 5: Foundation models in practice*